# F1 Track Limits — Dynamic Camera Segmentation

This notebook:
1. Pulls your annotated `inside` polygon dataset from Roboflow
2. Trains a YOLOv8 **instance segmentation** model on it
3. Runs the trained model **independently on every frame** of a broadcast/onboard clip

Because inference is done per-frame (no homography, no tracker state carried between frames), a constantly panning/zooming/cutting camera doesn't break anything — every frame gets a fresh segmentation of the `inside` track polygon.

**Before running:** In Colab, go to `Runtime > Change runtime type` and select a GPU.

In [ ]:
!pip install -q roboflow ultralytics opencv-python-headless

## Step 1 — Download your annotated dataset from Roboflow

Get your API key from `https://app.roboflow.com/settings/api`. Use the `yolov8` export format — it exports polygon masks (not just boxes), which is what YOLOv8-seg needs.

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "YOUR_API_KEY"   # <-- fill in
WORKSPACE = "your-workspace"        # <-- fill in
PROJECT = "your-project"            # <-- fill in
VERSION = 1                          # <-- the dataset version you generated with augmentations

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

print("Downloaded to:", dataset.location)

In [ ]:
import yaml

with open(f"{dataset.location}/data.yaml") as f:
    data_cfg = yaml.safe_load(f)

print(data_cfg)
assert "inside" in data_cfg["names"], "'inside' class not found in data.yaml — check your export"

## Step 2 — Train YOLOv8-seg

`yolov8n-seg.pt` is the fastest/smallest starting point. If accuracy on curbs/run-off edges is too low, swap in `yolov8s-seg.pt` or `yolov8m-seg.pt`.

`imgsz=960` is set higher than the default 640 since broadcast frames are widescreen and track-limit boundaries (curbs, white lines) are thin, high-frequency details that benefit from more resolution.

In [ ]:
from ultralytics import YOLO

seg_model = YOLO("yolov8n-seg.pt")

results = seg_model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=960,
    batch=8,
    patience=20,
    project="f1-track-limits",
    name="inside-seg",
    device=0,
)

In [ ]:
# This validates against the 'valid' split, the same split used for early stopping
# during training (see 'patience' above). It tells you how training went, but it's
# not an unbiased estimate of real-world performance since it influenced training.
metrics = seg_model.val()
print(metrics.seg)

### Holdout check on the `test` split

Roboflow already carved out a `test/` folder when you generated the dataset version (separate from `train/` and `valid/`). This set was never touched during training or early stopping, so it's your honest estimate of how the model performs on unseen data. Always check this before trusting the model on new race footage.

**Note on split sizing for a 250-image dataset:** Roboflow's default is 70/20/10 (train/valid/test), which with 250 base images leaves only ~25 test images and ~50 validation images before augmentation multiplies the train set. That's thin for reliably catching failure modes (e.g. a specific curb color, a rain-soaked surface, a camera angle you don't have much of). If you haven't generated the version yet, consider:
- Making sure test/valid images are spread across different circuits, lighting conditions, and camera types rather than randomly sampled (Roboflow's random split won't guarantee this on its own)
- Leaning slightly more images into valid/test than the default if your 250 images cover very different scene types

If you already generated the version with a split you're unhappy with, you'll need to regenerate the version in Roboflow with new percentages — the split is fixed at generation time, not something this notebook can change after the fact.

In [ ]:
test_metrics = seg_model.val(
    data=f"{dataset.location}/data.yaml",
    split="test",
)
print(test_metrics.seg)

## Step 3 — Upload your dynamic-camera clip

Any mp4/mov works. This is the clip where the camera angle/zoom keeps changing (broadcast cuts, onboard cam, TV pan/zoom, etc.).

In [ ]:
from google.colab import files

uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print("Uploaded:", video_path)

## Step 4 — Run per-frame segmentation on the clip

Key idea for handling the changing camera: **there is no tracking, no optical flow, and no fixed homography between frames.** Each frame is passed to the model independently, so a cut to a different camera, a zoom, or a pan doesn't require any special handling — the model just re-segments the `inside` polygon fresh on whatever it sees.

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

BEST_WEIGHTS = "f1-track-limits/inside-seg/weights/best.pt"
model = YOLO(BEST_WEIGHTS)

CONF_THRES = 0.4
ALPHA = 0.35                 # mask overlay transparency
INSIDE_CLASS_NAME = "inside"

class_names = model.names
inside_class_id = [k for k, v in class_names.items() if v == INSIDE_CLASS_NAME][0]

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out_path = "track_limits_output.mp4"
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    result = model.predict(frame, conf=CONF_THRES, verbose=False)[0]

    overlay = frame.copy()
    inside_contour = None

    if result.masks is not None:
        for mask_xy, cls_id in zip(result.masks.xy, result.boxes.cls):
            if int(cls_id) != inside_class_id:
                continue

            poly = mask_xy.astype(np.int32)
            cv2.fillPoly(overlay, [poly], color=(0, 255, 0))

            # Keep the largest 'inside' polygon detected on this frame
            if inside_contour is None or cv2.contourArea(poly) > cv2.contourArea(inside_contour):
                inside_contour = poly

    frame = cv2.addWeighted(overlay, ALPHA, frame, 1 - ALPHA, 0)

    if inside_contour is not None:
        cv2.polylines(frame, [inside_contour], isClosed=True, color=(0, 200, 255), thickness=2)

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()
print(f"Processed {frame_idx} frames -> {out_path}")

## Step 5 (optional) — Track-limits violation check

Once you also have a car detector (bounding box or a point representing the car's contact patch), you can test whether that point falls outside the `inside` polygon on that frame. This is a placeholder you can wire into the loop above once you add a car-detection class/model.

In [ ]:
def point_outside_track(point, inside_contour):
    """
    point: (x, y) pixel coordinate, e.g. a car's rear-contact-patch or centroid
    inside_contour: the largest 'inside' polygon detected on this frame (or None)

    Returns:
        True  -> point is outside the inside boundary (potential track-limits violation)
        False -> point is inside the boundary
        None  -> no 'inside' polygon was detected on this frame, can't judge
    """
    if inside_contour is None:
        return None
    result = cv2.pointPolygonTest(inside_contour, point, False)
    return result < 0

# Example usage once you have a car detector feeding coordinates:
# car_center = (car_x, car_y)
# violated = point_outside_track(car_center, inside_contour)

## Step 6 — Download the annotated output video

In [ ]:
from google.colab import files

files.download(out_path)